# Arena 3DGS — 84-Image Reconstruction

Uses a pre-computed COLMAP model (global mapper, 84/84 images registered, 18.5K points, outliers removed).
Upload the zip file created by `scripts/export_3dgs_input.py`.

**Steps:**
| Step | Cell | What | Time | GPU? |
|------|------|------|------|------|
| 1 | **Cell 1** | Mount Drive + start/continue session | 30s | No |
| 2 | **Cell 2** | Install deps (COLMAP, PyTorch, gsplat) | 3 min | No* |
| 3 | **Cell 3** | Upload & extract COLMAP zip | 1 min | No |
| 4 | **Cell 4** | Convert to 3DGS format | 1 min | No |
| 5 | **Cell 5** | Train 3DGS (30K iters, ~30 min) | 30 min | **Yes (T4+)** |
| 6 | **Cell 6** | Export PLY + validate + compress | 1 min | No |

## Session Management

On resume, data is restored from Drive so subsequent steps can continue.
---
## Setup
---

In [ ]:
#@title === 1. Mount Drive + Session Management ===
import os, sys

SCRIPTS_DIR = "/content/scripts"
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

# Download modules from GitHub if on Colab
import urllib.request
os.makedirs(SCRIPTS_DIR, exist_ok=True)
for mod in ["session.py", "colab_pipeline.py"]:
    dest = os.path.join(SCRIPTS_DIR, mod)
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/kaarthik-balakrishnan/arena-3dgs/main/scripts/{mod}"
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"Downloaded {mod}")
        except Exception as e:
            print(f"Could not download {mod}: {e}")

from scripts.session import Session, get_drive_path

DRIVE_PATH = get_drive_path()
session = Session(DRIVE_PATH)
existing = session.load()

if existing["created_at"] is None:
    existing["created_at"] = __import__('datetime').datetime.now().isoformat()
    session.save(existing)
    print("\nStarting fresh session.")
else:
    done_steps = [k for k, v in existing["steps"].items() if v]
    print(f"\nExisting session found with {len(done_steps)} completed step(s):")
    for s in done_steps:
        print(f"    - {s}")
    print("\nOn resume, data will be restored from Drive.")

RESET = False  #@param {type:"boolean"}
if RESET:
    session.reset()
    print("\nSession reset. All steps will re-run.")


In [ ]:
#@title === 2. Install Dependencies (~3 min, idempotent) ===
from scripts.colab_pipeline import install_dependencies
install_dependencies(session)


---
## Step 3: Upload COLMAP Data (~1 min)

Upload the zip file (`arena_3dgs_input.zip`) produced by `scripts/export_3dgs_input.py`.
On resume, data is restored from Drive.
---

In [ ]:
#@title === 3. Upload & Extract COLMAP Data (~1 min, idempotent) ===
import os, zipfile, glob, shutil

INPUT_DIR = "/content/gaussian-splatting/input"
SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")

if session.is_step_done("colmap_extracted"):
    print("COLMAP data already extracted. Restoring from Drive...")
    session.restore_from_drive("arena_input", INPUT_DIR)

os.makedirs(SPARSE_DIR, exist_ok=True)

img_txt = os.path.join(SPARSE_DIR, "images.txt")
if os.path.exists(img_txt):
    with open(img_txt) as f:
        n = sum(1 for l in f if l.strip() and not l.startswith("#")) // 2
    print(f"Found existing model: {n} images")
    if n >= 80:
        session.mark_step("colmap_extracted")
        session.save_to_drive(INPUT_DIR, "arena_input")
    else:
        print(f"Only {n} images - looking for 84-image zip...")

if not session.is_step_done("colmap_extracted"):
    candidates = glob.glob("/content/*.zip") + glob.glob("/tmp/*.zip") + glob.glob("*.zip")
    if not candidates:
        raise FileNotFoundError(
            "No .zip file found in /content/.\n"
            "Upload arena_3dgs_input.zip via Colab left sidebar -> Files -> upload to /content/"
        )

    zip_path = candidates[0]
    print(f"Extracting {os.path.basename(zip_path)} ({os.path.getsize(zip_path)/1024/1024:.0f} MB)...")

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(INPUT_DIR)

    nested = os.path.join(INPUT_DIR, "arena_3dgs_input")
    if os.path.exists(nested):
        for item in os.listdir(nested):
            shutil.move(os.path.join(nested, item), os.path.join(INPUT_DIR, item))
        os.rmdir(nested)
        print("  Flattened nested directory.")

    images_dir = os.path.join(INPUT_DIR, "images")
    if not os.path.exists(images_dir) or len(os.listdir(images_dir)) < 80:
        for root, dirs, files in os.walk(INPUT_DIR):
            jpgs = [f for f in files if f.lower().endswith(".jpg")]
            if len(jpgs) >= 80:
                os.makedirs(images_dir, exist_ok=True)
                for f in jpgs:
                    shutil.move(os.path.join(root, f), os.path.join(images_dir, f))
                print(f"  Moved {len(jpgs)} images to input/images/")
                break

    with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
        n = sum(1 for l in f if l.strip() and not l.startswith("#")) // 2
    pts_file = os.path.join(SPARSE_DIR, "points3D.txt")
    n_pts = 0
    if os.path.exists(pts_file):
        n_pts = sum(1 for l in open(pts_file) if l.strip() and not l.startswith("#"))
    img_count = len(os.listdir(os.path.join(INPUT_DIR, "images"))) if os.path.exists(os.path.join(INPUT_DIR, "images")) else 0

    print(f"\n  {n} registered images")
    print(f"  {n_pts} sparse points")
    print(f"  {img_count} image files")

    if n < 80:
        raise RuntimeError(f"Expected 80+ images in model, got {n}. Wrong zip?")

    session.set_param("training_images", n)
    session.mark_step("colmap_extracted")
    # Upload extracted data to Drive for resume
    session.save_to_drive(INPUT_DIR, "arena_input")


---
## Step 4: Convert to 3DGS Format (~1 min)
---

In [ ]:
#@title === 4. Convert to 3DGS Format (~1 min, idempotent) ===
from scripts.colab_pipeline import convert_to_3dgs_format
convert_to_3dgs_format(session)


---
## Step 5: Train 3D Gaussian Splatting

Requires GPU (T4 or better). Training uses gsplat for 2x faster rendering.
---

In [ ]:
#@title === 5A: Quick Test (~7 min) ===
from scripts.colab_pipeline import train_3dgs
train_3dgs(session, iterations=3000, max_gaussians=100000, log_interval=500, max_res=800, output_name="arena_3dgs")


In [ ]:
#@title === 5B: Full Training 30K (~30 min, single run) ===
from scripts.colab_pipeline import train_3dgs
train_3dgs(session, iterations=30000, max_gaussians=500000, log_interval=1000, max_res=800, output_name="arena_3dgs")


---
## Step 6: Export & Compress
---

In [ ]:
#@title === 6A: Export Final PLY (~1 min) ===
import os, shutil, glob

OUTPUT_DIR = "/content/gaussian-splatting/output/arena_3dgs"
COLAB_PLY = "/content/arena_3dgs_pointcloud.ply"
DRIVE_PLY = os.path.join(DRIVE_PATH, "arena_3dgs_pointcloud.ply")

plys = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.ply")))
if not plys:
    print("No PLY found in output. Train first (Cell 5A or 5B).")
else:
    latest = plys[-1]
    print(f"Latest PLY: {latest} ({os.path.getsize(latest)/1024/1024:.0f} MB)")
    shutil.copy2(latest, COLAB_PLY)
    print(f"Copied to {COLAB_PLY}")
    shutil.copy2(latest, DRIVE_PLY)
    print(f"Copied to {DRIVE_PLY} (Drive backup)")
    from google.colab import files
    files.download(COLAB_PLY)
    print("\nDownload started. Open Cell 6B-6C for validation and compression.")

In [ ]:
#@title === 6B: Validate PLY Format ===
from scripts.colab_pipeline import validate_pointcloud
validate_pointcloud("/content/arena_3dgs_pointcloud.ply")


In [ ]:
#@title === 6C: Compress for Local Viewer (~1 min) ===
import os, sys, subprocess, urllib.request

COLAB_PLY = "/content/arena_3dgs_pointcloud.ply"
if not os.path.exists(COLAB_PLY):
    print(f"PLY not found: {COLAB_PLY}. Run Cell 6A first.")
else:
    COMPRESS_PY = "/content/scripts/compress_splat.py"
    if not os.path.exists(COMPRESS_PY):
        print("Downloading compress script...")
        url = ("https://raw.githubusercontent.com/"
               "kaarthik-balakrishnan/arena-3dgs/main/scripts/compress_splat.py")
        urllib.request.urlretrieve(url, COMPRESS_PY)

    for quality in ['medium']:
        out_name = COLAB_PLY.replace('.ply', f'_{quality}.splat')
        !python "{COMPRESS_PY}" "{COLAB_PLY}" --quality {quality} --output "{out_name}"
        if os.path.exists(out_name):
            print(f"Compressed: {out_name} ({os.path.getsize(out_name)/1024/1024:.1f} MB)")
            from google.colab import files
            files.download(out_name)